# Derive Security Table

This notebook pulls live IBKR portfolio positions and combines them with curated security-reference metadata plus fresh IBKR contract details.

Prerequisites:
- Run it with the `py313` kernel.
- Start TWS or IB Gateway locally.
- Enable API access and confirm the host, port, and client ID match your local settings.
- Each live notebook or script needs a unique `IB_CLIENT_ID`.
- If the optional single-contract lookup is ambiguous, tighten `PRIMARY_EXCHANGE` or set `CONID` directly.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
while not (project_root / "market_helper").exists():
    if project_root.parent == project_root:
        raise RuntimeError("Could not find project root containing the market_helper package.")
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

SYMBOL = "XLK"
SEC_TYPE = "STK"
EXCHANGE = "SMART"
PRIMARY_EXCHANGE = "ARCA"
CURRENCY = "USD"
CONID = None
LOCAL_SYMBOL = None

IB_HOST = "127.0.0.1"
IB_PORT = 7497
IB_CLIENT_ID = 121
IB_ACCOUNT = ""

print(f"Project root: {project_root}")
print(
    "Optional single-contract lookup: "
    f"symbol={SYMBOL}, sec_type={SEC_TYPE}, exchange={EXCHANGE}, "
    f"primary_exchange={PRIMARY_EXCHANGE}, currency={CURRENCY}, conid={CONID}"
)
print(f"IBKR connection: host={IB_HOST}, port={IB_PORT}, client_id={IB_CLIENT_ID}, account={IB_ACCOUNT or '<default>'}")

In [ ]:
from pprint import pprint

from IPython.display import display

try:
    import pandas as pd
except ModuleNotFoundError:
    pd = None

from market_helper.domain.portfolio_monitor import build_live_ibkr_position_security_table

enriched_rows = build_live_ibkr_position_security_table(
    host=IB_HOST,
    port=IB_PORT,
    client_id=IB_CLIENT_ID,
    account_id=IB_ACCOUNT or None,
)

print(
    f"Pulled {len(enriched_rows)} portfolio rows with position metrics, "
    "security-reference enrichment, and IBKR contract details."
)

if pd is None:
    enriched_positions = enriched_rows
    pprint(enriched_rows[:3])
else:
    enriched_positions = pd.DataFrame(enriched_rows)
    if enriched_positions.empty:
        display(enriched_positions)
    else:
        preview_columns = [
            "account",
            "internal_id",
            "symbol",
            "local_symbol",
            "quantity",
            "market_value",
            "latest_price",
            "unrealized_pnl",
            "security_mapping_status",
            "security_display_ticker",
            "security_display_name",
            "security_report_category",
            "security_risk_bucket",
            "contract_long_name",
            "contract_primary_exchange",
        ]
        display(
            enriched_positions.loc[:, preview_columns]
            .sort_values("market_value", ascending=False, na_position="last")
            .reset_index(drop=True)
        )

In [ ]:
from pprint import pprint

from market_helper.data_sources.ibkr.tws import TwsIbAsyncClient

lookup_client = TwsIbAsyncClient(
    host=IB_HOST,
    port=IB_PORT,
    client_id=IB_CLIENT_ID,
    account=IB_ACCOUNT,
)

try:
    lookup_client.connect()
    security_info = lookup_client.lookup_security(
        symbol=SYMBOL,
        sec_type=SEC_TYPE,
        exchange=EXCHANGE,
        primary_exchange=PRIMARY_EXCHANGE,
        currency=CURRENCY,
        conid=CONID,
        local_symbol=LOCAL_SYMBOL,
    )
finally:
    lookup_client.disconnect()

pprint(security_info)

In [ ]:
from pprint import pprint

if isinstance(enriched_positions, list):
    if not enriched_positions:
        print("No enriched portfolio rows were returned.")
        pprint(enriched_positions)
    else:
        selected_row = next((row for row in enriched_positions if row.get("symbol") == SYMBOL), None)
        if selected_row is None:
            print(f"No combined row matched SYMBOL={SYMBOL}; showing the first portfolio row instead.")
            selected_row = enriched_positions[0]
        pprint(selected_row)
else:
    if enriched_positions.empty or "symbol" not in enriched_positions.columns:
        print("No enriched portfolio rows were returned.")
        display(enriched_positions)
    else:
        selected_rows = enriched_positions.loc[enriched_positions["symbol"].eq(SYMBOL)]
        if selected_rows.empty:
            print(f"No combined row matched SYMBOL={SYMBOL}; showing the first portfolio row instead.")
            display(enriched_positions.head(1).T)
        else:
            display(selected_rows.head(1).T)
